## MoE

- shared experts
- fine-grained experts
- router loss (2 variations)


nn.Sequential - automatically the input flows through the specified layers  
nn.ModuleList - you need to iterate through thel layers with a for loop


In [2]:
import torch.nn as nn
import torch
import torch.nn.functional as F
from einops import rearrange

class Expert(nn.Module):

	def __init__(self, d_ff:int, d_embed:int):
		super().__init__()

		self.network = nn.Sequential( 
			nn.Linear(d_embed, d_ff), 
			nn.ReLU(),
			nn.Dropout(0.1), # hidden layer dropout, always after non-linearity, since ReLU 
			nn.Linear(d_ff, d_embed),
			nn.Dropout(0.1), # right before residual
		)

	def forward(self, x_in): 

		x_out = self.network(x_in) # residual stream gets added outside the block

		return x_out

class Router(nn.Module): 

	def __init__(self, num_experts:int, d_embed:int): 
		super().__init__()
		self.router = nn.Linear(d_embed, num_experts)

	def forward(self, x_in): 

		x_out = self.router(x_in)
		return x_out

MoE Block


A nice way to visualize what an MoE block is doing

<img src="https://awsdocs-neuron.readthedocs-hosted.com/en/latest/_images/moe-architecture-overview.png" width="600">

- x_in = [N tokens, d_embed] <- token input
- [token indexes,] <- which tokens have been assigned to this expert i
  - we get this by doing passing the input through a router, to give us -> [N tokens, num_experts]
  - then we get the topk experts -> [N tokens, topk]
  - from that, for token 1 we get [expert 1, expert 2] for example
  - then we find all the indices in this tensor where our expert i, is one of those topk - this would mean that token should pass thru this expert
  - [N tokens, relevant topk experts] == expert i, along dim=1, which gives a boolean matrix back that is the same size
  - Identify which tokens there is a True boolean for
  - non.zero() gives us the exact indices of each token where a True exists!
  - this returns us tensor[token_idx], tensor[ranking in the topk experts chosen for this token] -> dim=1 can also be interpreted as the expert index for that token

_Now its easy_

- use the [token indexes, ] tensor (AKA: token_idx) to pluck out all the tokens assigned to expert i from out input x_in
- GETTING THE EXPERT OUTPUTS:
  - now pass that batched input through expert(x_flat)
  - this is the batching of the tokens for expert i (see: BLOCK)

<img src="https://developer-blogs.nvidia.com/wp-content/uploads/2025/10/image6-png.webp" width="600">

- GETTING THE EXPERT WEIGHTS:
  - use the same token_idx tensor and the tensor[expert i's ranking for each selected token], to pluck out the softmaxxed logits given by the router (i.e. probability assigned to this expert for this token)
- WHAT WE HAVE NOW: [Relevant tokens to expert i, probability weight for each of these tokens], [Relevant tokens to expert i, expert i's output for each of these tokens]

- FINISHING MOVE
  - now multiply the expert outputs by the probability weights, and then store that output. That output is the contribution of expert i, to all N token's output values

_Now repeat_

- do this but loop over all experts since each will contribute to every token's output value


In [ ]:
class MoeBlock(nn.Module):

	def __init__(self, num_experts:int, d_embed:int, d_ff:int, topk:int, num_shared_experts:int=None): # all defaults must be at the end of init
		super().__init__()

		self.num_experts = num_experts

		# initialize the router 
		self.router = Router(num_experts=num_experts, d_embed=d_embed)

		# initialize the experts
		self.experts = nn.ModuleList([Expert(d_ff=d_ff, d_embed=d_embed) for _ in range(num_experts)])

		# top_k routing
		self.topk = topk

		# shared experts - `or 0` so the None default means "no shared experts" instead of crashing on range(None)
		self.shared_experts = nn.ModuleList([Expert(d_ff=d_ff, d_embed=d_embed) for _ in range(num_shared_experts or 0)])
		
	def forward(self, x_in):

		B, T, D = x_in.shape

		# use the router to find which experts should be routed to, then softmax 
		router_logits = self.router(x_in)

		full_router_probs = F.softmax(router_logits, dim=-1) # used to compute loss

		logit_values, score_indices = torch.topk(router_logits, k=self.topk) # returns a tuple

		# select the topk
		score_values = F.softmax(logit_values, dim=-1) # shaped (B, T, topk)
		
		# using the topk array, using the indices, index into the experts Module list, and pass x_in through that. Then sum all together, weighting the output by the value of the topk output
		# for this type of operation use x.index_add_(dim, index, source, *, alpha=1)
		
		# initialize the array to store each token
		x_flat = x_in.view(B*T, D) # (N d)
		idx_flat = score_indices.view(B*T, self.topk) # (N, topk) <- indices
		gates_flat = score_values.view(B*T, self.topk) # (N, topk) <- values
		
		out = torch.zeros((B * T, D)).to(x_in.device)

		# ideally, you would like to index into the expert array with the last_dimension of score_indices, creating a (B, T, top_k) matrix. However this would require replicating
		# the expert tensors which would be bad for memory, so instead we iteratively cycle through the experts, which hold the tensor that is expensive to hold in memory
		# it also makes sense to accumulate all the tokens assigned to an expert and do a batch matmul

		fraction = torch.zeros(len(self.experts)) # (N_experts,)
		probs = full_router_probs.flatten(0,1).mean(dim=0) # mean probs per expert across all tokens (N_experts,)

		for e, expert in enumerate(self.experts):

			# check if expert is one of the top_k
			token_idx, slot_idx = (idx_flat == e).nonzero(as_tuple=True) 
			# .nonzero output -> the columns store the index for dim0, dim1, so to index correctly you must zip the tensors
			# token_idx = dim0 token index, slot_idx = dim1 expert index, tensors are size (len(token_idx),)

			if token_idx.numel() == 0: # this expert does not attend to any tokens
				continue

			# now index into x_flat to pull out the tokens
			tokens = x_flat[token_idx] # (L, d_embed), where L = len(token_idx)

			# ok so now we can feed the tokens into the experts
			# but that output needs to be weighted by the score_values
			weights = gates_flat[token_idx, slot_idx] # (L,) <- this should hold the respective weights for each token assigned to this expert

			out.index_add_(dim=0, index=token_idx, source=weights.unsqueeze(-1) * expert(tokens)) # index in the seq_len dimension, weights get broadcasted and multiply along the dimensions

			fraction[e] = len(slot_idx) / (B*T*self.topk) # how many tokens were assigned to expert e // total number of possible slots

		# an empty ModuleList is falsy, so this loop simply does not run when there are no shared experts
		for expert in self.shared_experts: 
			
			# add shared experts! 
			out += expert(x_flat)

		moe_out = out.view(B, T, D)

		return moe_out, len(self.experts) * torch.sum(fraction * probs)

# Find the variance along the expert dimension -> (B, T, num_experts), sum across B, T and average

x_in = torch.rand(32, 1024, 256)
moeblock = MoeBlock(num_experts=12, d_ff=128, d_embed=256, topk=2, num_shared_experts=2)

x_out = moeblock(x_in)

MoE GPT


In [7]:
class RotaryEmbeddings(nn.Module): 

	def __init__(self, d_embed: int, max_seq_len: int, base: int=1e4):

		super().__init__()

		assert d_embed % 2 == 0

		positions = torch.arange(max_seq_len).unsqueeze(-1) # (max_seq_len, 1)

		pair_idx = torch.arange(0, d_embed, 2)

		inv_freq = torch.exp(-math.log(base) * pair_idx / d_embed) # (d_embed//2,)

		angles = positions * inv_freq # (max_seq_len, d_embed//2), outer product

		sin = torch.sin(angles)
		cos = torch.cos(angles)

		self.register_buffer('sin', sin, persistent=False)
		self.register_buffer('cos', cos, persistent=False)

	def forward(self, x_in):

		T = x_in.shape[1]

		x_even = x_in[..., 0::2] # (B, T, d_embed // 2)
		x_odd = x_in[..., 1::2]

		# rotation matrix is [[cosx, -sinx],[sinx, cosx]] * [x_even, x_odd]
		out_even = x_even * self.cos[:T] - x_odd * self.sin[:T]
		out_odd = x_even * self.sin[:T] + x_odd * self.cos[:T]

		return torch.stack((out_even, out_odd), dim=-1).flatten(-2)

In [ ]:
import math 

# copy of MLA for the GPT code
# TODO: implement deepseek-v3 sparse attention
class MLA(nn.Module): 

	def __init__(self, q_latent_dim:int=12, kv_latent_dim:int=4, d_embed:int=256, num_heads:int=8, max_seq_len:int=1024): 
		super().__init__()

		self.q_latent = nn.Linear(d_embed, q_latent_dim)
		self.query = nn.Linear(q_latent_dim, d_embed)
		self.kv_latent = nn.Linear(d_embed, kv_latent_dim)
		self.key = nn.Linear(kv_latent_dim, d_embed)
		self.value = nn.Linear(kv_latent_dim, d_embed)
		self.num_heads = num_heads

		self.pos_embedding = RotaryEmbeddings(d_embed=d_embed, max_seq_len=max_seq_len)

		self.w_out = nn.Linear(d_embed, d_embed)
		
		self.register_buffer('causal_mask', torch.tril(torch.ones(max_seq_len, max_seq_len)).bool(), persistent=False) # this is a nn.Module method that registers a self.causal_mask but on CUDA

	def forward(self, x_in:int, block_kv_cache=None): 
		is_prefill = block_kv_cache is None

		b, t, d = x_in.shape

		q_latent = self.q_latent(x_in)
		query_latent = self.query(q_latent)
		query = rearrange(query_latent, 'b q (n d) -> b n q d', n=self.num_heads)

		kv_latent = self.kv_latent(x_in)

		if not is_prefill:
			# torch.cat is terrible because it creates new memory
			kv_latent = torch.cat((block_kv_cache['kv_latent'], kv_latent), dim=1) # (B, T, low_rank) -> concat along the seq_len dimension
			# then (B, T, low_rank) * (low_rank, d_embed) -> (B, T, d_embed) which reassembles the full key, value matrices
		
		# reassemble full kv matrices (costs O(2*latent_dim*embed_dim))
		key_latent = self.key(kv_latent)
		value_latent = self.value(kv_latent)
		
		# reshape into heads
		key = rearrange(key_latent, 'b k (n d) -> b n k d', n=self.num_heads)
		value = rearrange(value_latent, 'b v (n d) -> b n v d', n=self.num_heads)

		# assemble causal self-attention matrix
		logits = torch.einsum('b n q d, b n k d -> b n q k', query, key) / ((d//self.num_heads)**0.5) # divide by the head_dim of the

		if is_prefill:
			logits = logits.masked_fill(~self.causal_mask[:t, :t], float('-inf')) # where the masked_fill isn't true, replace it with -inf

		scores = torch.softmax(logits, dim=-1)
		attention = torch.einsum('b n q k, b n k d -> b n q d', scores, value)
		attention = rearrange(attention, 'b n q d -> b q (n d)')
		
		attention_output = self.w_out(attention)

		block_kv_cache = {
			"kv_latent": kv_latent,
		}
		
		return attention_output, block_kv_cache

<img src="https://substackcdn.com/image/fetch/$s_!W4Qo!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Ff4b97110-d705-4531-b4af-4f87187a8dea_1393x1394.png" width="400">


In [ ]:
from dataclasses import dataclass
from collections import defaultdict
import torch
import torch.nn as nn

@dataclass
class MoeConfig:

	d_embed: int = 256
	d_ff: int = 1024
	num_experts: int = 64
	topk: int = 4
	num_transformer_blocks: int = 12
	num_shared_experts: int = 2
	max_seq_len: int = 1024
	num_heads: int = 8
	vocab_size: int= 50257


class MoeTransformer(nn.Module): 

	def __init__(self, d_embed:int, d_ff:int, num_experts:int, topk:int, num_shared_experts:int=None, block_kv_cache=None, max_seq_len:int=1024, num_heads:int=8):
		super().__init__()
		
		self.moe_block = MoeBlock(num_experts=num_experts, d_embed=d_embed, d_ff=d_ff, topk=topk, num_shared_experts=num_shared_experts)
		self.mla = MLA(d_embed=d_embed, max_seq_len=max_seq_len, num_heads=num_heads)

		# we'll use pre-norm as per DSv3
		self.ln1 = nn.LayerNorm(d_embed)
		self.ln2 = nn.LayerNorm(d_embed)

	def forward(self, x_in, block_kv_cache=None):

		norm_x_in = self.ln1(x_in) # first layernorm to pass into att
		attention, kv_cache = self.mla(norm_x_in, block_kv_cache=block_kv_cache) # att

		x_r1 = x_in + attention # residual connection 1
		
		norm_x_r1 = self.ln2(x_r1) # normalize before moe
		out_moe, lb_loss = self.moe_block(norm_x_r1) # moe output

		x_r2 = out_moe + x_r1 # residual connection 2

		return x_r2, kv_cache, lb_loss


class MoeGPT(nn.Module):

	def __init__(self, d_embed:int, d_ff:int, num_experts:int, topk:int, num_transformer_blocks: int, num_shared_experts:int=None, max_seq_len:int=1024, num_heads:int=8, vocab_size:int=50257):
		super().__init__()

		self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_embed)

		self.blocks = nn.ModuleList([MoeTransformer(d_embed=d_embed, d_ff=d_ff, num_experts=num_experts, topk=topk, num_shared_experts=num_shared_experts, max_seq_len=max_seq_len, num_heads=num_heads) for _ in range(num_transformer_blocks)])

		self.lm_head = nn.Linear(d_embed, vocab_size, bias=False) # out projection layer, converting embeddings into tokens
		self.lm_head.weight = self.embedding.weight # weight tying matters for small models, where an embedding layer is ~13M parameters

		self.max_seq_len = max_seq_len

		self.block_vars = defaultdict(list)

	# teacher forcing
	def forward(self, x_in: int, targets: list[int]=None):

		B, T = x_in.shape

		assert T <= self.max_seq_len

		# never tokenize in the forward, tokenize in the data_loader
		# input_ids = self.tokenizer(x_in) # (B, T)
		# input_ids = input_ids["input_ids"]

		x_in = self.embedding(x_in) # (B, T, D)

		total_lb_loss = 0.0
		for i, block in enumerate(self.blocks):
			x_in, kv_cache, lb_loss = block(x_in)
			total_lb_loss += lb_loss
		
		logits = self.lm_head(x_in) # (B, T, vocab_size)


		loss = None
		if targets is not None: 
			shifted_logits = logits[:, :-1, :].flatten(0,1) # (N, vocab_size)
			shifted_targets = targets[:, 1:].reshape(-1,1) # (N,)

			log_probs = F.log_softmax(shifted_logits, dim=-1) # prevents numerical overflow when you softmax, then log(0.00001), (N, vocab_size)
			# gather index must match source index tensor shape
			target_log_probs = log_probs.gather(dim=-1, index=shifted_targets.reshape(-1, 1)).squeeze(-1) # index into the last dim, pluck out the target token_id then squeeze -> (N,)
			loss = -target_log_probs.mean(dim=1) # 1/N sum(-log(q(token)))

		return logits, loss

data = "I really like eating icecream, its my favorite thing in the entire world"

self.tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V3", trust_remote_code=True)

x_in

config = MoeConfig()
config = vars(config)
moe = MoeGPT(**config)

x_out, loss = moe(x_in)
print(x_out.shape, loss)

KeyboardInterrupt: 

Deepseek Router

- significantly weakens the weight of the auxiliary loss (from 0.01 to 0.0001)
- uses a bias term instead that tries to balance out expert tokens
- first finds the best M experts, locates the other experts on their nodes, then selects topk from those. Guarantees only M nodes are involved in the MoE layer, so less communication overhead


torch.scatter\_ - index dimensions must be the same as the input dimensions (tensor you're doing operation on), aside from the dimension you're indexing into which can be different but ≤

- e.g. x.scatter\_(dim=1, index=indices, value=1)
  - x = B, T, D
  - index = B, tokens I want to index into, D -> lets say the tokens are [5,6,7]
  - x = B, T, D (editted in place)
  - here I am adding +1 to every token's embeddings, for the tokens 5,6,7

torch.gather - input tensor is B, T, D, index tensor is whatever you want the output shape to be

- e.g. out = torch.gather(x, dim=-1, index)
  - x = B, T, num_experts
  - index = B, T, topk
  - out = B, T, topk - plucked out the gathered tokens


In [ ]:
class DeepseekRouter(nn.Module): 

	def __init__(self, num_experts=256, topk=8, top_m_nodes=4, experts_per_node=64, gamma=0.001):

		super().__init__()

		assert num_experts % experts_per_node == 0

		self.router = nn.Linear(d_embed, num_experts)
		self.sigmoid = nn.Sigmoid()

		# register buffer moves it to device, normal tensors are not moved by .to(device), nor will they be saved with the model checkpoints
		self.register_buffer('bias', torch.zeros(num_experts)) # (N,), persistent=True because we want this buffer to be checkpointed
		
		# in Deepseek's router, it also accounts for expert parallelism - where many experts are stored on a single node
		# we want to route to as few nodes as possible to prevent communication overhead between nodes - to all-reduce the output tensor
		# so instead of selecting top_k, we'll select top M first (top 4 experts), then we'll select the rest of our topk from experts on those respective nodes
		self.experts_per_node = experts_per_node
		self.num_nodes = num_experts // experts_per_node
		self.num_experts = num_experts
		self.topk = topk

	def forward(self, x_in):

		B, T, _ = x_in.shape

		logits = router(x_in)
		raw_affinities = sigmoid(logits) # crushes between 0 and 1

		# scores for each expert
		selection_scores = raw_affinities + self.bias # (B, T, num_experts)

		# assume that experts are organized such that the first chunk of experts along the expert dimension belong to node 1, then the second to node 2, etc.
		node_grouped_scores = selection_scores.view(-1, self.num_nodes, self.experts_per_node) # (B*T, num_nodes, experts_per_node)

		node_max_scores, _ = node_grouped_scores.max(dim=-1) # .max() collapses along the specified dimension, meaning we are finding the best expert in each node
		# (B*T, num_nodes, experts_per_node) 

		_, top_nodes_idx = torch.topk(node_max_scores, self.top_m_nodes, dim=-1) # select the topM experts, given the best expert on each node - this gives us the best top_m nodes 
		# (B*T, top_m)

		# mask out all experts on nodes other than these
		valid_experts_mask = torch.zeros_like(node_grouped_scores, dtype=torch.bool) # (B*T, num_nodes, experts_per_node), all initialized to false by default since zeros
		# keep only the experts on the top_m nodes valid 
		valid_experts_mask.scatter_(dim=1, index=top_nodes_idx.unsqueeze(-1).expand(-1,-1, self.experts_per_node).contiguous(), value=True) # (B*T, top_m, experts_per_node) <- this indexes into dim1 of (B, T, num_nodes, experts_per_node)
		valid_experts_mask = valid_experts_mask.view(-1, self.num_experts) # (B*T, num_nodes, experts_per_node) -> (B*T, num_experts)

		# masking True, all of the nodes and their respective experts that are valid
		selection_scores = selection_scores.masked_fill(~valid_experts_mask, float('-inf')) # where true, masked to -inf, (B*T, num_experts)
		
		# then we select the topk from those 4 nodes
		_, topk_expert_indices = torch.topk(selection_scores, self.topk, dim=-1) # index into the nodes layer, then topk along all the experts along each node
		# (B*T, topk)

		# given the topk experts, we want to return the raw_affinities (before the bias was applied)
		gates = torch.gather(input=raw_affinities.view(-1, num_experts), dim=-1, index=topk_expert_indices) # (B*T, topk)
		gates = gates / gates.sum(dim=-1, keepdim=True) # normalize the topk into probabilities

		return gates.view(B, T, self.topk), topk_expert_indices.view(B, T, self.topk)

	def update_bias(self, selected_topk_indices: torch.Tensor):

		B, T, K = selected_topk_indices.shape

		with torch.no_grad():
			# self.bias -> (num_experts,)
			# selected_topk_experts -> (B, T, topk), index into dim=-1 and add there
			experts = torch.zeros(B*T, self.num_experts, device=selected_topk_indices.device)
			experts.scatter_(dim=1, index=selected_topk_indices.view(-1, K), value=1.0) # (B*T, num_experts), dim=1 because I want to add to the num_experts dim not token dim

			expert_load = experts.sum(dim=0, keepdim=False) / (B*T) # (num_experts,)

			mean_load = self.topk / self.num_experts

			self.bias += self.gamma * torch.sign(expert_load - mean_load)

Latent MoE (Kimi K3)
